# Lecture 2: Orthogonal Vectors and Matrices
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Adjoint (Conjugate Transpose) vs. Transpose

In the complex case, the adjoint $A^*$ conjugates *and* transposes. Plain transpose $A^T$ only transposes.

In [ ]:
np.random.seed(0)
A = np.random.rand(2, 4) + 1j * np.random.rand(2, 4)
print("A =")
print(A)

print("\nAdjoint (conjugate transpose) A.conj().T =")
print(A.conj().T)

print("\nPlain transpose A.T =")
print(A.T)

## Inner Products

For complex column vectors $u, v$, the inner product is $u^* v$. The result is a scalar.

In [ ]:
u = np.array([4, -1, 2+2j])
v = np.array([-1, 1j, 1])

innerprod = np.vdot(u, v)
print("u =", u)
print("v =", v)
print("\nInner product u* v =", innerprod)

### Length via the 2-norm

$\|u\|^2 = u^* u$

In [ ]:
length_u_sq = np.vdot(u, u)
print("|u|^2 = u* u =", length_u_sq)
print("sum(|u_i|^2)  =", np.sum(np.abs(u)**2))
print("norm(u)        =", np.linalg.norm(u))

### Angle between vectors

$\cos\theta = \frac{u^* v}{\|u\|\,\|v\|}$

The angle may be complex when the vectors are complex!

In [ ]:
cos_theta = np.vdot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
print("cos(theta) =", cos_theta)

theta = np.arccos(cos_theta)
print("theta      =", theta)

### Inverse and Hermitian commute

$(A^{-1})^* = (A^*)^{-1}$, so we can write $A^{-*}$ unambiguously.

In [ ]:
np.random.seed(1)
A = np.random.rand(4, 4) + 1j * np.random.rand(4, 4)

inv_then_adjoint = np.linalg.inv(A).conj().T
adjoint_then_inv = np.linalg.inv(A.conj().T)

print("max |inv(A)* - inv(A*)| =", np.max(np.abs(inv_then_adjoint - adjoint_then_inv)))

## Orthogonality and Orthonormal Columns

We generate a matrix $Q$ with orthonormal columns via the (thin) QR factorization.

$Q^* Q = I$ confirms orthonormality.

In [ ]:
np.random.seed(2)
Q, _ = np.linalg.qr(np.random.rand(5, 3))

print("Q (5x3 with orthonormal columns):")
print(np.round(Q, 4))

print("\nQ* Q (should be I):")
print(np.round(Q.conj().T @ Q, 10))

## Orthogonal Projection and Decomposition

Given a vector $u$, compute coefficients $c = Q^* u$, projection $v = Qc$, and residual $r = u - v$.

The residual $r$ is orthogonal to every column of $Q$.

In [ ]:
u = np.random.rand(5)

c = Q.conj().T @ u
print("Coefficients c = Q* u =", c)

v = Q @ c
print("\nProjection v = Q c =", v)

r = u - v
print("\nResidual r = u - v =", r)

In [ ]:
# Verify: r is orthogonal to columns of Q
print("Q* r (should be ~0):")
print(Q.conj().T @ r)

# Verify: v and r are orthogonal
print("\nv . r (should be ~0):", np.vdot(v, r))

## Unitary Matrices

A square matrix with orthonormal columns is **unitary**: $Q^* Q = I$ and $Q^* = Q^{-1}$.

In [ ]:
np.random.seed(3)
Q_full, _ = np.linalg.qr(np.random.rand(5, 5) + 1j * np.random.rand(5, 5))

# inv(Q) should equal Q*
diff = np.abs(np.linalg.inv(Q_full) - Q_full.conj().T)
print("max |inv(Q) - Q*| =", np.max(diff), " (should be ~0)")

In [ ]:
# With a square unitary Q, the projection residual is zero
c = Q_full.conj().T @ u
v = Q_full @ c
r = u - v
print("Residual r (should be ~0):")
print(r)

## Preservation of Length

If $Q$ is orthogonal/unitary, $\|Qx\| = \|x\|$ for every $x$.

**Proof:** $\|Qx\|^2 = x^* Q^* Q x = x^* x = \|x\|^2$.

In [ ]:
# Rotation by 90 degrees
Q_rot = np.array([[0, -1],
                   [1,  0]], dtype=float)

x = np.array([3.0, 4.0])
Qx = Q_rot @ x

print(f"x  = {x},  ||x||  = {np.linalg.norm(x)}")
print(f"Qx = {Qx}, ||Qx|| = {np.linalg.norm(Qx)}")

In [ ]:
# Visualize: rotation preserves length
fig, ax = plt.subplots(figsize=(5, 5))

ax.annotate('', xy=x, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.annotate('', xy=Qx, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

circle = plt.Circle((0, 0), np.linalg.norm(x), fill=False,
                     linestyle='--', color='gray', alpha=0.5)
ax.add_patch(circle)

ax.text(x[0] + 0.2, x[1], 'x = (3, 4)', color='blue', fontsize=11)
ax.text(Qx[0] - 2.5, Qx[1] + 0.3, 'Qx = (-4, 3)', color='red', fontsize=11)

ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_title('90° rotation preserves length')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Numerical Stability: Orthogonal vs. Ill-Conditioned

An orthogonal matrix has condition number 1. An ill-conditioned matrix amplifies errors.

In [ ]:
# Ill-conditioned matrix
A_bad = np.array([[1, 1],
                   [0, 1e-10]])

print(f"cond(A_bad)  = {np.linalg.cond(A_bad):.2e}")
print(f"cond(Q_rot)  = {np.linalg.cond(Q_rot):.2e}")

In [ ]:
# Demonstrate error amplification
b_exact = np.array([2.0, 1e-10])
x_exact = np.linalg.solve(A_bad, b_exact)

# Perturb b by a tiny amount
b_noisy = b_exact + np.array([0, 1e-16])
x_noisy = np.linalg.solve(A_bad, b_noisy)

print(f"Perturbation in b: {np.linalg.norm(b_noisy - b_exact):.2e}")
print(f"Perturbation in x: {np.linalg.norm(x_noisy - x_exact):.2e}")
print(f"Amplification:     {np.linalg.norm(x_noisy - x_exact) / np.linalg.norm(b_noisy - b_exact):.2e}")

## Orthogonal Projection: Geometric Visualization

Project a 3D vector onto the $xy$-plane.

In [ ]:
v = np.array([3.0, 4.0, 5.0])

# Orthonormal basis for xy-plane
e1 = np.array([1.0, 0.0, 0.0])
e2 = np.array([0.0, 1.0, 0.0])

v_hat = np.dot(e1, v) * e1 + np.dot(e2, v) * e2
residual = v - v_hat

print(f"v       = {v}")
print(f"v_hat   = {v_hat}  (projection onto xy-plane)")
print(f"residual = {residual}  (perpendicular to plane)")
print(f"v_hat . residual = {np.dot(v_hat, residual)}  (orthogonal check)")

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')

origin = [0, 0, 0]

# Original vector
ax.quiver(*origin, *v, color='blue', arrow_length_ratio=0.08, linewidth=2, label='v = (3,4,5)')
# Projection
ax.quiver(*origin, *v_hat, color='green', arrow_length_ratio=0.1, linewidth=2, label=r'$\hat{v}$ = (3,4,0)')
# Residual
ax.quiver(*v_hat, *residual, color='red', arrow_length_ratio=0.15, linewidth=2, label='r = (0,0,5)')
# Dashed line connecting v to projection
ax.plot([v[0], v_hat[0]], [v[1], v_hat[1]], [v[2], v_hat[2]],
        'k--', alpha=0.4)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('Orthogonal projection onto the xy-plane')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## General Rotation Matrix

A 2D rotation by angle $\theta$ is orthogonal: $Q^TQ = I$, $\|Qx\| = \|x\|$.

In [ ]:
theta = np.pi / 6  # 30 degrees
Q_theta = np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])

print("Q (30° rotation):")
print(np.round(Q_theta, 4))
print("\nQ^T Q =")
print(np.round(Q_theta.T @ Q_theta, 10))
print("\ncond(Q) =", np.linalg.cond(Q_theta))

In [ ]:
# Visualize rotation applied to many vectors on the unit circle
angles = np.linspace(0, 2 * np.pi, 100)
circle_pts = np.array([np.cos(angles), np.sin(angles)])
rotated_pts = Q_theta @ circle_pts

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(circle_pts[0], circle_pts[1], 'b-', alpha=0.3, label='Original')
ax.plot(rotated_pts[0], rotated_pts[1], 'r--', alpha=0.3, label=f'Rotated {int(np.degrees(theta))}°')

# Show a specific vector
x_demo = np.array([1.0, 0.0])
Qx_demo = Q_theta @ x_demo
ax.annotate('', xy=x_demo, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.annotate('', xy=Qx_demo, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_title(f'Rotation by {int(np.degrees(theta))}° preserves the unit circle')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()